# Rotas mais curtas por equipe

Para cada uma das 49 equipes:
1. Filtra pacientes pelo bbox da área de cobertura (drop dos outliers de anonimização).
2. Particiona os pacientes da equipe em **K clusters geográficos** (k-means em lat/lon), simulando divisão por agente.
3. Para cada cluster, chama o OSRM local (perfil `foot`) com `/trip` partindo da UBS, `roundtrip=true`.
4. Acumula rotas num DataFrame e visualiza no mapa.

## ⚠️ Caveat

Dataset anonimizado: ruído gaussiano de ~100m nas coords + shuffle de endereço dentro do território da equipe + outliers espúrios (~7%, já filtrados). **As rotas geradas aqui são exercício metodológico, não roteiro operacional.** Indicadores absolutos (km, minutos) não refletem realidade — só dinâmica relativa entre equipes/clusters.

Além disso, o OSM mapeia bem ruas urbanas formais, mas vielas/escadarias/passagens em comunidades são frequentemente incompletas — rotas devem ser tratadas como sugestão metodológica.

## Setup

In [ ]:
from pathlib import Path
import json
import time

import duckdb
import pandas as pd
import requests
import folium
from sklearn.cluster import KMeans

OSRM_FOOT = "http://localhost:5000"
BBOX = {"lon_min": -43.310, "lat_min": -22.995, "lon_max": -43.180, "lat_max": -22.885}
DATA_DIR = Path.home() / "Documents"
K_DEFAULT = 8  # clusters por equipe (panel hipotético de 1 ACS ~250 pacientes)
MIN_PACIENTES_POR_CLUSTER = 20
RANDOM_STATE = 42

## Health-check OSRM
Se falhar: `cd routing && docker compose --profile routing up -d osrm-foot`.

In [ ]:
def osrm_health():
    r = requests.get(
        f"{OSRM_FOOT}/route/v1/foot/-43.234,-22.928;-43.252,-22.913?overview=false",
        timeout=5,
    )
    r.raise_for_status()
    code = r.json().get("code")
    assert code == "Ok", f"OSRM retornou code={code}"
    print(f"OSRM foot OK ({OSRM_FOOT})")

osrm_health()

## Carga dos parquets + filtro de outliers

In [ ]:
con = duckdb.connect()

equipes = con.execute(f"""
    SELECT equipe_id,
           endereco_latitude  AS ubs_lat,
           endereco_longitude AS ubs_lon
    FROM read_parquet('{DATA_DIR}/equipes_anonimizadas.parquet')
""").df()

pacientes_all = con.execute(f"""
    SELECT paciente_id, equipe_id,
           endereco_latitude  AS lat,
           endereco_longitude AS lon
    FROM read_parquet('{DATA_DIR}/pacientes_anonimizados.parquet')
    WHERE endereco_latitude  IS NOT NULL
      AND endereco_longitude IS NOT NULL
""").df()

in_bbox = (
    pacientes_all["lon"].between(BBOX["lon_min"], BBOX["lon_max"]) &
    pacientes_all["lat"].between(BBOX["lat_min"], BBOX["lat_max"])
)
pacientes = pacientes_all[in_bbox].reset_index(drop=True)

print(f"equipes:           {len(equipes)}")
print(f"pacientes total:   {len(pacientes_all):>7,}")
print(f"pacientes no bbox: {len(pacientes):>7,}  ({len(pacientes)/len(pacientes_all):.1%})")
print(f"equipes com >=1 paciente no bbox: {pacientes['equipe_id'].nunique()}")

## Clusterização por equipe (k-means em lat/lon)

In [ ]:
def clusterizar(grupo: pd.DataFrame, k: int = K_DEFAULT) -> pd.DataFrame:
    n = len(grupo)
    k_eff = max(1, min(k, n // MIN_PACIENTES_POR_CLUSTER or 1))
    if k_eff == 1:
        grupo = grupo.copy()
        grupo["cluster_id"] = 0
        return grupo
    km = KMeans(n_clusters=k_eff, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(grupo[["lat", "lon"]].values)
    grupo = grupo.copy()
    grupo["cluster_id"] = labels
    return grupo

pacientes_clust = (
    pacientes.groupby("equipe_id", group_keys=False)
    .apply(lambda g: clusterizar(g, K_DEFAULT))
    .reset_index(drop=True)
)

resumo = (
    pacientes_clust.groupby("equipe_id")
    .agg(n_pacientes=("paciente_id", "size"), n_clusters=("cluster_id", "nunique"))
    .reset_index()
)
print(resumo.describe())
resumo.head()

## Roteamento OSRM por (equipe, cluster)

In [ ]:
def osrm_trip(coords: list[tuple[float, float]]) -> dict | None:
    """coords = [(lon, lat), ...] com UBS na posição 0. Retorna dict do OSRM ou None se falhar."""
    coord_str = ";".join(f"{lon},{lat}" for lon, lat in coords)
    url = f"{OSRM_FOOT}/trip/v1/foot/{coord_str}"
    params = {
        "source": "first",
        "roundtrip": "true",
        "geometries": "geojson",
        "overview": "full",
    }
    try:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"[ERR] {e}")
        return None
    if data.get("code") != "Ok":
        print(f"[WARN] OSRM code={data.get('code')} message={data.get('message')}")
        return None
    return data


def rotear_equipe_cluster(equipe_id, ubs_lat, ubs_lon, sub: pd.DataFrame):
    coords = [(ubs_lon, ubs_lat)] + list(zip(sub["lon"], sub["lat"]))
    pids = [None] + sub["paciente_id"].tolist()
    data = osrm_trip(coords)
    if data is None:
        return None, None
    trip = data["trips"][0]
    total_dur = trip["duration"]
    geom = trip["geometry"]
    wps = data["waypoints"]
    rows = []
    for idx_in_coords, wp in enumerate(wps):
        rows.append({
            "equipe_id":         equipe_id,
            "cluster_id":        int(sub["cluster_id"].iloc[0]),
            "ordem":             wp["waypoint_index"],
            "paciente_id":       pids[idx_in_coords],
            "is_ubs":            idx_in_coords == 0,
            "lon":               coords[idx_in_coords][0],
            "lat":               coords[idx_in_coords][1],
            "total_duration_s":  total_dur,
        })
    return pd.DataFrame(rows).sort_values("ordem").reset_index(drop=True), geom


rotas_rows = []
geoms = {}  # (equipe_id, cluster_id) -> GeoJSON geometry
ubs_lookup = equipes.set_index("equipe_id")[["ubs_lat", "ubs_lon"]].to_dict("index")

t0 = time.time()
for (eq, cl), sub in pacientes_clust.groupby(["equipe_id", "cluster_id"]):
    if eq not in ubs_lookup:
        print(f"[SKIP] equipe {eq} sem UBS no parquet de equipes")
        continue
    ubs = ubs_lookup[eq]
    df_rota, geom = rotear_equipe_cluster(eq, ubs["ubs_lat"], ubs["ubs_lon"], sub)
    if df_rota is None:
        continue
    rotas_rows.append(df_rota)
    geoms[(eq, cl)] = geom

rotas = pd.concat(rotas_rows, ignore_index=True)
print(f"\n{len(geoms)} rotas em {time.time()-t0:.1f}s")
rotas.head()

## Resumo agregado

In [ ]:
por_equipe = (
    rotas[~rotas["is_ubs"]]
    .groupby("equipe_id")
    .agg(
        n_clusters=("cluster_id", "nunique"),
        n_pacientes=("paciente_id", "nunique"),
    )
)
duracao_por_cluster = (
    rotas.groupby(["equipe_id", "cluster_id"])["total_duration_s"]
    .first()
    .reset_index()
)
por_equipe["horas_total"] = (
    duracao_por_cluster.groupby("equipe_id")["total_duration_s"].sum() / 3600
)
por_equipe.sort_values("horas_total", ascending=False).head(10)

In [ ]:
ax = duracao_por_cluster["total_duration_s"].div(3600).hist(bins=50, figsize=(8, 4))
ax.set_xlabel("horas por cluster (round-trip)")
ax.set_ylabel("# clusters")
ax.set_title("Distribuição de duração por cluster");

## Visualização no mapa
`plot_equipe(equipe_id)` desenha a UBS, todos os pacientes coloridos por cluster, e a polyline OSRM de cada rota.

In [ ]:
PALETTE = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#46f0f0", "#f032e6", "#bcf60c", "#fabebe",
    "#008080", "#9a6324", "#800000", "#808000", "#000075",
]


def plot_equipe(equipe_id: str) -> folium.Map:
    sub = pacientes_clust[pacientes_clust["equipe_id"] == equipe_id]
    if sub.empty:
        raise ValueError(f"equipe {equipe_id} sem pacientes no bbox")
    ubs = ubs_lookup[equipe_id]
    m = folium.Map(location=[ubs["ubs_lat"], ubs["ubs_lon"]], zoom_start=14, tiles="CartoDB positron")

    folium.Marker(
        location=[ubs["ubs_lat"], ubs["ubs_lon"]],
        icon=folium.Icon(color="black", icon="plus-sign"),
        popup=f"UBS equipe {equipe_id[:8]}",
    ).add_to(m)

    for _, p in sub.iterrows():
        color = PALETTE[int(p["cluster_id"]) % len(PALETTE)]
        folium.CircleMarker(
            location=[p["lat"], p["lon"]],
            radius=3, color=color, fill=True, fill_opacity=0.7, weight=1,
        ).add_to(m)

    for cl in sorted(sub["cluster_id"].unique()):
        geom = geoms.get((equipe_id, cl))
        if geom is None:
            continue
        color = PALETTE[int(cl) % len(PALETTE)]
        dur_s = rotas[(rotas["equipe_id"] == equipe_id) & (rotas["cluster_id"] == cl)]["total_duration_s"].iloc[0]
        latlon = [(lat, lon) for lon, lat in geom["coordinates"]]
        folium.PolyLine(
            locations=latlon, color=color, weight=3, opacity=0.7,
            tooltip=f"cluster {cl} — {dur_s/3600:.1f}h round-trip",
        ).add_to(m)

    return m

equipe_amostra = por_equipe.sort_values("horas_total", ascending=False).index[0]
print(f"equipe amostra: {equipe_amostra}")
plot_equipe(equipe_amostra)